# NEOSHOP — YOLOv8 Training
**100% self-contained. Uses only pre-installed Colab packages.**
Run cells top to bottom. ~25 minutes total.

## Cell 1 — Check GPU

In [ ]:
!nvidia-smi
import torch
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print('OK')
else:
    print('ERROR — Runtime > Change runtime type > T4 GPU')

## Cell 2 — Install Only What We Need

In [ ]:
!pip install ultralytics mediapipe -q
print('Done')

## Cell 3 — Download COCO val2017
Official COCO server — always available. This gives us both product images AND hand images.

In [ ]:
import urllib.request

print('Downloading COCO val2017 images (~800MB)...')
urllib.request.urlretrieve(
    'http://images.cocodataset.org/zips/val2017.zip',
    '/content/coco_val.zip'
)
print('Extracting...')
!unzip -q /content/coco_val.zip -d /content/coco

print('Downloading annotations...')
urllib.request.urlretrieve(
    'http://images.cocodataset.org/annotations/annotations_trainval2017.zip',
    '/content/coco_ann.zip'
)
!unzip -q /content/coco_ann.zip -d /content/coco

import os
n = len(os.listdir('/content/coco/val2017'))
print(f'Done — {n} images available')

## Cell 4 — Extract Product Labels from COCO
Bottles, cups, apples, bananas, oranges, carrots, bowls, cake, book, scissors

In [ ]:
import json, shutil, random
from pathlib import Path
from collections import defaultdict

# COCO category ID -> (our class id, name)
# class 0 = hand (added later via MediaPipe)
GROCERY_MAP = {
    44: (1,  'bottle'),
    47: (2,  'cup'),
    51: (3,  'bowl'),
    52: (4,  'banana'),
    53: (5,  'apple'),
    55: (6,  'orange'),
    57: (7,  'carrot'),
    60: (8,  'cake'),
    84: (9,  'book'),
    86: (10, 'scissors'),
    90: (11, 'toothbrush'),
}

OUT = Path('/content/dataset')
for split in ['train', 'valid']:
    (OUT / split / 'images').mkdir(parents=True, exist_ok=True)
    (OUT / split / 'labels').mkdir(parents=True, exist_ok=True)

with open('/content/coco/annotations/instances_val2017.json') as f:
    coco = json.load(f)

id2file = {img['id']: img['file_name'] for img in coco['images']}
id2size = {img['id']: (img['width'], img['height']) for img in coco['images']}

img_anns = defaultdict(list)
for ann in coco['annotations']:
    if ann['category_id'] in GROCERY_MAP:
        img_anns[ann['image_id']].append(ann)

product_images = []  # track which images have products (for MediaPipe hand pass later)
saved = 0
for img_id, anns in img_anns.items():
    fname = id2file[img_id]
    W, H  = id2size[img_id]
    src   = Path('/content/coco/val2017') / fname
    if not src.exists():
        continue
    yolo_lines = []
    for ann in anns:
        cls_id, _ = GROCERY_MAP[ann['category_id']]
        x, y, bw, bh = ann['bbox']
        xc = (x + bw/2) / W
        yc = (y + bh/2) / H
        bw /= W
        bh /= H
        yolo_lines.append(
            f'{cls_id} {max(0,min(1,xc)):.6f} {max(0,min(1,yc)):.6f} '
            f'{max(0.01,min(1,bw)):.6f} {max(0.01,min(1,bh)):.6f}'
        )
    if not yolo_lines:
        continue
    split = 'train' if random.random() < 0.85 else 'valid'
    dst_img = OUT / split / 'images' / fname
    shutil.copy(src, dst_img)
    with open(OUT / split / 'labels' / f'{Path(fname).stem}.txt', 'w') as f:
        f.write('\n'.join(yolo_lines))
    product_images.append(str(src))
    saved += 1

print(f'Saved {saved} product images')

## Cell 5 — Auto-Label Hands Using MediaPipe
We run MediaPipe on ALL COCO images to find ones containing hands.
MediaPipe gives us precise hand bounding boxes automatically — no manual labeling needed.
This is the most reliable method possible: no external download, runs entirely in Colab.

In [ ]:
import cv2
import mediapipe as mp
import numpy as np
from pathlib import Path
import random, shutil

OUT = Path('/content/dataset')
COCO_IMGS = Path('/content/coco/val2017')

mp_hands   = mp.solutions.hands
all_images = sorted(COCO_IMGS.glob('*.jpg'))

print(f'Running MediaPipe on {len(all_images)} COCO images...')
print('This finds all images containing hands and auto-generates labels.')
print('Takes about 5-8 minutes...')

hand_saved = 0
checked    = 0

with mp_hands.Hands(
    static_image_mode=True,
    max_num_hands=4,
    min_detection_confidence=0.5
) as hands:
    for img_path in all_images:
        checked += 1
        if checked % 500 == 0:
            print(f'  Checked {checked}/{len(all_images)} — found {hand_saved} hand images so far')

        img_bgr = cv2.imread(str(img_path))
        if img_bgr is None:
            continue
        H, W = img_bgr.shape[:2]
        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        result  = hands.process(img_rgb)

        if not result.multi_hand_landmarks:
            continue

        yolo_lines = []
        for hand_landmarks in result.multi_hand_landmarks:
            xs = [lm.x for lm in hand_landmarks.landmark]
            ys = [lm.y for lm in hand_landmarks.landmark]
            # Add padding around the hand
            pad = 0.05
            x1  = max(0.0, min(xs) - pad)
            y1  = max(0.0, min(ys) - pad)
            x2  = min(1.0, max(xs) + pad)
            y2  = min(1.0, max(ys) + pad)
            xc  = (x1 + x2) / 2
            yc  = (y1 + y2) / 2
            bw  = x2 - x1
            bh  = y2 - y1
            if bw > 0.02 and bh > 0.02:
                yolo_lines.append(f'0 {xc:.6f} {yc:.6f} {bw:.6f} {bh:.6f}')

        if not yolo_lines:
            continue

        # Check if this image already exists in dataset (has product labels)
        # If so, APPEND hand labels to existing label file
        added = False
        for split in ['train', 'valid']:
            dst_img = OUT / split / 'images' / img_path.name
            dst_lbl = OUT / split / 'labels' / f'{img_path.stem}.txt'
            if dst_img.exists():
                with open(dst_lbl, 'a') as f:
                    f.write('\n' + '\n'.join(yolo_lines))
                added = True
                break

        if not added:
            # New image (not in product set) — add as hand-only image
            split    = 'train' if random.random() < 0.85 else 'valid'
            dst_img  = OUT / split / 'images' / img_path.name
            dst_lbl  = OUT / split / 'labels' / f'{img_path.stem}.txt'
            shutil.copy(img_path, dst_img)
            with open(dst_lbl, 'w') as f:
                f.write('\n'.join(yolo_lines))

        hand_saved += 1

print(f'\nDone!')
print(f'Hand images found and labeled: {hand_saved}')
print(f'Train images: {len(list((OUT/"train/images").glob("*")))}')
print(f'Valid images: {len(list((OUT/"valid/images").glob("*")))}')

## Cell 6 — Write data.yaml

In [ ]:
import yaml
from pathlib import Path

OUT = Path('/content/dataset')

CLASS_NAMES = [
    'hand',        # 0  — from MediaPipe auto-label
    'bottle',      # 1  — from COCO
    'cup',         # 2
    'bowl',        # 3
    'banana',      # 4
    'apple',       # 5
    'orange',      # 6
    'carrot',      # 7
    'cake',        # 8
    'book',        # 9
    'scissors',    # 10
    'toothbrush',  # 11
]

train_n = len(list((OUT/'train/images').glob('*')))
valid_n = len(list((OUT/'valid/images').glob('*')))

print(f'Dataset summary:')
print(f'  Train: {train_n} images')
print(f'  Valid: {valid_n} images')
print(f'  Classes ({len(CLASS_NAMES)}): {CLASS_NAMES}')

with open(OUT / 'data.yaml', 'w') as f:
    yaml.dump({
        'path':  str(OUT),
        'train': 'train/images',
        'val':   'valid/images',
        'nc':    len(CLASS_NAMES),
        'names': CLASS_NAMES
    }, f, default_flow_style=False)

print('data.yaml written — ready to train')

## Cell 7 — Train YOLOv8n (~20 minutes)

In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8n.pt')

results = model.train(
    data='/content/dataset/data.yaml',
    epochs=60,
    imgsz=640,
    batch=16,
    device=0,
    name='neoshop_v1',
    project='/content/runs',
    patience=15,
    save=True,
    plots=True,
    fliplr=0.5,
    mosaic=1.0,
    degrees=10,
    scale=0.5,
    hsv_s=0.7,
    hsv_v=0.4,
    translate=0.1,
)

print('Training complete!')
print(f'mAP50: {results.results_dict["metrics/mAP50(B)"]:.3f}')
print('Model: /content/runs/neoshop_v1/weights/best.pt')

## Cell 8 — Download best.pt

In [ ]:
from google.colab import files
files.download('/content/runs/neoshop_v1/weights/best.pt')
print('Done!')
print('Put best.pt into: neoshop_cv/models/best.pt')
print('Then run: python 2_theft_detection.py')